# B2-020-language-transformers — Practice p23 — Solution

**Type:** challenge · **Difficulty:** advanced · **Concepts:** language-transformer, causal-language-modeling

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).

## Solution

Correct trace: IDs `(3,8)` become inputs and targets `(3,7)` by `inputs=tokens[:,:-1]`, `targets=tokens[:,1:]`; hidden is `(3,7,8)`; a vocabulary head `Linear(8,12)` produces logits `(3,7,12)`; loss uses `logits.reshape(-1,12)` against `targets.reshape(-1)` while ignoring padding. The reported `(3,12,8)` swaps vocabulary and sequence axes, and its unshifted targets are an independent label-alignment bug.

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)

class TinyEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(12, 8, padding_idx=0)
        self.position_embedding = nn.Embedding(8, 8)
        self.norm1 = nn.LayerNorm(8, eps=1e-5)
        self.attention = nn.MultiheadAttention(8, 2, dropout=0.0, batch_first=True)
        self.norm2 = nn.LayerNorm(8, eps=1e-5)
        self.ff1 = nn.Linear(8, 16)
        self.ff2 = nn.Linear(16, 8)

    def forward(self, token_ids, *, mask_mode):
        length = token_ids.shape[1]
        positions = torch.arange(length, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        normalized = self.norm1(x)
        attention_mask = None
        if mask_mode == "causal":
            attention_mask = torch.triu(
                torch.ones(length, length, dtype=torch.bool, device=token_ids.device),
                diagonal=1,
            )
        elif mask_mode != "bidirectional":
            raise ValueError(f"unknown mask mode: {mask_mode}")
        attended, _ = self.attention(
            normalized,
            normalized,
            normalized,
            attn_mask=attention_mask,
            key_padding_mask=token_ids.eq(0),
            need_weights=False,
        )
        x = x + attended
        return x + self.ff2(F.gelu(self.ff1(self.norm2(x))))

torch.manual_seed(20260812)
tokens = torch.tensor([[2,4,6,8,10,3,0,0], [2,5,7,8,11,3,0,0], [2,4,7,9,10,3,0,0]], dtype=torch.int64)
inputs = tokens[:, :-1]
targets = tokens[:, 1:]
encoder = TinyEncoder(); vocab_head = nn.Linear(8, 12)
hidden = encoder(inputs, mask_mode="causal")
logits = vocab_head(hidden)
loss = F.cross_entropy(logits.reshape(-1, 12), targets.reshape(-1), ignore_index=0)
mutated_inputs = inputs.clone(); mutated_inputs[:, 5] = torch.tensor([9,4,5])
mutated_logits = vocab_head(encoder(mutated_inputs, mask_mode="causal"))

### Answer check

In [ ]:
assert inputs.shape == targets.shape == (3, 7)
assert hidden.shape == (3, 7, 8) and logits.shape == (3, 7, 12)
assert targets[0, 0].item() == 4 and targets[0, -1].item() == 0
assert loss.ndim == 0
assert torch.allclose(logits[:, :5], mutated_logits[:, :5], atol=1e-6, rtol=1e-6)